In [ ]:
from ultralytics import YOLO
from ultralytics.utils.metrics import box_iou

import supervision as sv
from supervision.metrics import MeanAveragePrecision
from ensemble_boxes import weighted_boxes_fusion

from pathlib import Path
import random


import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import cv2 as cv

import torch

## Baseline YOLO26s run without custom augmentations (for comparison with the custom-aug run)

In [ ]:
model=YOLO('yolo26s.pt')

results = model.train( data="/mnt/d/dataset/helmet_dataset/data.yaml",
                       epochs=50, patience=15, imgsz=640,
                       batch=8, device=0, cache="ram", seed=0,
                       name="s_no_custom_aug")

In [ ]:
img = Image.open('/home/ziba/runs/detect/s_no_custom_aug/results.png')
dpi = 150
fig = plt.figure(figsize=(img.width / dpi, img.height / dpi), dpi=dpi)
ax = fig.add_axes([0, 0, 1, 1])      # axes تمام figure را پر کند
ax.imshow(img)
ax.axis("off")
fig.savefig("assets/training_curves.png", dpi=dpi)

## Evaluate the YOLO model on the validation set and print precision, recall, mAP50, and mAP50-95.

In [ ]:
model_path_s='/home/ziba/runs/detect/s_no_custom_aug/weights/best.pt'

r=YOLO(model_path_s).val(data='/mnt/d/dataset/helmet_dataset/data.yaml', split="val", imgsz=640, plots=False)

print('model_s', f"P={r.box.mp:.3f} R={r.box.mr:.3f} mAP50={r.box.map50:.3f} mAP50-95={r.box.map:.3f}")

## Load the valid split into supervision and get class names

In [ ]:
ds=sv.DetectionDataset.from_yolo(images_directory_path='/mnt/d/dataset/helmet_dataset/valid/images',
                                 annotations_directory_path='/mnt/d/dataset/helmet_dataset/valid/labels',
                                 data_yaml_path='/mnt/d/dataset/helmet_dataset/data.yaml')
class_names=ds.classes

## Load the best-trained YOLO model without custom augmentation.

In [ ]:
m = YOLO('/home/ziba/runs/detect/s_no_custom_aug/weights/best.pt')

## Evaluate all four YOLO model checkpoints on the validation set and compare their precision, recall, and mAP scores.

In [ ]:
weights = {"n_baseline": '/home/ziba/runs/detect/train/weights/best.pt',
           "n_fixed":    "/home/ziba/runs/detect/train_fixed_annotation_loooooooooooosened/weights/best.pt",
           "n_aug":      "/home/ziba/runs/detect/train_loosened_pluuuus_aug/weights/best.pt",
           "s":          "/home/ziba/runs/detect/s_no_custom_aug/weights/best.pt",}

for name, w in weights.items():
    r = YOLO(w).val(data="/mnt/d/dataset/helmet_dataset/data.yaml", split="val", imgsz=640, plots=False)
    print(name, f"P={r.box.mp:.3f} R={r.box.mr:.3f} mAP50={r.box.map50:.3f} mAP50-95={r.box.map:.3f}")

## Threshold sweep (P/R/FN/FP) for one model; used to compare the three trained models

In [ ]:
def threshold_table(model_path, ts=(0.05, 0.1, 0.15, 0.25)):
    m = YOLO(model_path)
    preds, gts = [], []
    for _, img, label in ds:
        r = m.predict(img, imgsz=640, conf=0.001, iou=0.5, device=0, verbose=False)[0]
        preds.append(sv.Detections.from_ultralytics(r).with_nms())
        gts.append(label)
    for t in ts:
        pt = [d[d.confidence >= t] for d in preds]
        cm = sv.ConfusionMatrix.from_detections(pt, gts, classes=class_names,
                                                conf_threshold=t, iou_threshold=0.3)
        tp, fn, fp = cm.matrix[0, 0], cm.matrix[0, 1], cm.matrix[1, 0]
        print(f"conf≥{t}: P={tp/(tp+fp):.3f} R={tp/(tp+fn):.3f} FN={int(fn)} FP={int(fp)}")

        
print('=========YOLOvn with augmentation==============')
threshold_table('/home/ziba/runs/detect/train_loosened_pluuuus_aug/weights/best.pt')

print('=========YOLOvn without augmentation==============')
threshold_table('/home/ziba/runs/detect/train_fixed_annotation_loooooooooooosened/weights/best.pt')

print('=========YOLOvs without augmentation==============')
threshold_table('/home/ziba/runs/detect/s_no_custom_aug/weights/best.pt')

## TTA with WBF: run the model on several image variants (flip, lighting, brightness), map boxes back to original coordinates, and fuse them with Weighted Boxes Fusion

In [ ]:
identity = lambda d, w, h: d

def unflip(d, w, h):
    d.xyxy[:, [0, 2]] = w - d.xyxy[:, [2, 0]]
    return d

# each entry: (name, image_transform, box_inverse)
TRANSFORMS = { "orig":   (lambda im: im,                               identity),
               "flip":   (lambda im: im[:, ::-1].copy(),               unflip),
               "spot_l": (lambda im: add_spot_light(im, 0.25, 0.3),    identity),
               "spot_r": (lambda im: add_spot_light(im, 0.75, 0.3),    identity),
               "bright": (lambda im: cv.convertScaleAbs(im, alpha=1.0, beta=40),  identity),
               "dark":   (lambda im: cv.convertScaleAbs(im, alpha=1.0, beta=-40), identity),}

def predict_tta_wbf(m, img, names=("orig", "flip"), conf=0.001,
                    iou_thr=0.55, skip_thr=0.001, conf_type="avg"):
    h, w = img.shape[:2]
    boxes_list, scores_list, labels_list = [], [], []

    for n in names:
        tf, inv = TRANSFORMS[n]
        r = m.predict(tf(img), imgsz=640, conf=conf, iou=0.5, device=0, verbose=False)[0]
        d = inv(sv.Detections.from_ultralytics(r), w, h)
        b = d.xyxy.astype(np.float32).copy()
        b[:, [0, 2]] /= w
        b[:, [1, 3]] /= h
        boxes_list.append(np.clip(b, 0, 1))
        scores_list.append(d.confidence)
        labels_list.append(d.class_id)

    boxes, scores, labels = weighted_boxes_fusion(
        boxes_list, scores_list, labels_list,
        iou_thr=iou_thr, skip_box_thr=skip_thr, conf_type=conf_type)

    boxes[:, [0, 2]] *= w
    boxes[:, [1, 3]] *= h
    return sv.Detections(xyxy=boxes, confidence=scores, class_id=labels.astype(int))

## Add a synthetic Gaussian spotlight to the image: a soft bright blob centered at (cx_ratio, cy_ratio) of the image size, with spread and intensity set by radius_ratio and strength

In [ ]:
def add_spot_light(img, cx_ratio=0.5, cy_ratio=0.3, radius_ratio=0.4, strength=80):
    h, w = img.shape[:2]
    ys, xs = np.mgrid[0:h, 0:w]
    cx, cy = cx_ratio * w, cy_ratio * h
    sigma = radius_ratio * max(h, w)
    mask = np.exp(-((xs - cx) ** 2 + (ys - cy) ** 2) / (2 * sigma ** 2))
    out = img.astype(np.float32) + strength * mask[..., None]
    return np.clip(out, 0, 255).astype(np.uint8)

## TTA with NMS: predict on the original, horizontally flipped (boxes un-flipped), brighter, and darker versions, then merge all detections and remove duplicates with NMS

In [ ]:
def predict_tta(m, img, conf=0.001):
    h, w = img.shape[:2]

    def run(image):
        r = m.predict(image, imgsz=640, conf=conf, iou=0.5, device=0, verbose=False)[0]
        return sv.Detections.from_ultralytics(r)

    # 1. original
    outs = [run(img)]

    # 2. horizontal flip (boxes must be un-flipped)
    d = run(img[:, ::-1].copy())
    d.xyxy[:, [0, 2]] = w - d.xyxy[:, [2, 0]]
    outs.append(d)

    # 3. brighter / 4. darker (no box change needed)
    outs.append(run(cv.convertScaleAbs(img, alpha=1.0, beta=40)))
    outs.append(run(cv.convertScaleAbs(img, alpha=1.0, beta=-40)))

    return sv.Detections.merge(outs).with_nms(threshold=0.5)

## Evaluate WBF-based TTA configs: for each set of transforms, fuse predictions, then report P/R/FN/FP/total (FN+FP) at several confidence thresholds (IoU=0.3) in one DataFrame

In [ ]:
def eval_tta(m, names, ts=(0.05, 0.1, 0.15, 0.25), **kw):
    preds, gts = [], []
    for _, img, label in ds:
        preds.append(predict_tta_wbf(m, img, names=names, **kw))
        gts.append(label)
    rows = []
    for t in ts:
        pt = [d[d.confidence >= t] for d in preds]
        cm = sv.ConfusionMatrix.from_detections(pt, gts, classes=class_names, conf_threshold=t, iou_threshold=0.3)
        tp, fn, fp = cm.matrix[0, 0], cm.matrix[0, 1], cm.matrix[1, 0]
        rows.append({"tta": "+".join(names), "conf": t,
                     "P": round(tp / max(tp + fp, 1), 3), "R": round(tp / max(tp + fn, 1), 3),
                     "FN": int(fn), "FP": int(fp), "total": int(fn + fp)})
    return pd.DataFrame(rows)

configs = [("orig", "flip"),
           ("orig", "flip", "spot_l", "spot_r"),
           ("orig", "flip", "bright", "dark")]

results = pd.concat([eval_tta(m, c) for c in configs], ignore_index=True)
results

## Test-set evaluation config: dataset paths, inference settings (imgsz, device), confidence/NMS/matching IoU thresholds, and class names

In [ ]:
# dataset paths
DATA_YAML = "/mnt/d/dataset/helmet_dataset/data.yaml"

TEST_IMAGES_DIR = Path("/mnt/d/dataset/helmet_dataset/test/images")
TEST_LABELS_DIR = Path("/mnt/d/dataset/helmet_dataset/test/labels")

# Evaluation settings
IMGSZ = 640
DEVICE = 0

CONF_THRESHOLD = 0.10
NMS_IOU = 0.50
MATCH_IOU = 0.30

CLASS_NAMES = ["helmet"]

## Run Ultralytics' standard validation on the test split and print mean precision, recall, mAP@50 and mAP@50-95

In [ ]:
test_results = m.val( data=DATA_YAML, split="test", imgsz=IMGSZ, device=DEVICE, plots=True, verbose=True)

print("\n========== STANDARD VALIDATION METRICS ==========")
print(f"Precision : {test_results.box.mp:.4f}")
print(f"Recall    : {test_results.box.mr:.4f}")
print(f"mAP50     : {test_results.box.map50:.4f}")
print(f"mAP50-95  : {test_results.box.map:.4f}")

## Load YOLO-format ground-truth boxes from a label file and convert them to pixel xyxy coordinates (returns an empty (0, 4) array if the file is missing or has no boxes)

In [ ]:
def load_gt_boxes(label_path, image_width, image_height):
    """Load YOLO-format ground-truth boxes as pixel coordinates."""

    boxes = []

    if not label_path.exists():
        return np.empty((0, 4), dtype=float)

    with open(label_path, "r") as f:
        for line in f:
            values = line.strip().split()

            if len(values) < 5:
                continue

            cls, xc, yc, w, h = map(float, values[:5])

            x1 = (xc - w / 2) * image_width
            y1 = (yc - h / 2) * image_height
            x2 = (xc + w / 2) * image_width
            y2 = (yc + h / 2) * image_height

            boxes.append([x1, y1, x2, y2])

    return np.asarray(boxes, dtype=float).reshape(-1, 4)

## Compute IoU between two xyxy boxes (areas clamped at zero to handle degenerate boxes; returns 0.0 if the union is empty)

In [ ]:
def calculate_iou(box1, box2):
    """Calculate IoU between two xyxy boxes."""

    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])

    intersection = max(0, x2 - x1) * max(0, y2 - y1)

    area1 = max(0, box1[2] - box1[0]) * max(0, box1[3] - box1[1])
    area2 = max(0, box2[2] - box2[0]) * max(0, box2[3] - box2[1])

    union = area1 + area2 - intersection

    return intersection / union if union > 0 else 0.0

## Greedy one-to-one matching of GT and predicted xyxy boxes by highest IoU (>= match_iou); returns matched (gt, pred, iou) triples plus the indices of unmatched GT (FN) and unmatched predictions (FP)

In [ ]:
def match_predictions(gt_boxes, pred_boxes, match_iou=0.3):
    """Match predictions to GT boxes using greedy IoU matching."""

    candidates = []

    for gt_idx, gt_box in enumerate(gt_boxes):
        for pred_idx, pred_box in enumerate(pred_boxes):
            iou = calculate_iou(gt_box, pred_box)

            if iou >= match_iou:
                candidates.append((iou, gt_idx, pred_idx))

    candidates.sort(reverse=True)

    matched_gt = set()
    matched_pred = set()
    matches = []

    for iou, gt_idx, pred_idx in candidates:
        if gt_idx in matched_gt or pred_idx in matched_pred:
            continue

        matched_gt.add(gt_idx)
        matched_pred.add(pred_idx)

        matches.append((gt_idx, pred_idx, iou))

    fn_indices = [ i for i in range(len(gt_boxes)) if i not in matched_gt ]

    fp_indices = [ i for i in range(len(pred_boxes)) if i not in matched_pred]

    return matches, fn_indices, fp_indices

## Evaluate the model on the test set at a fixed confidence threshold: per-image greedy IoU matching to get TP/FP/FN, store visualization data, and print overall precision/recall

In [ ]:
image_paths = sorted( p for p in TEST_IMAGES_DIR.iterdir() if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp", ".webp"})

records = []
visual_data = {}

for image_path in image_paths:

    image = cv.imread(str(image_path))

    if image is None:
        print(f"Warning: could not read {image_path.name}")
        continue

    height, width = image.shape[:2]

    label_path = TEST_LABELS_DIR / f"{image_path.stem}.txt"
    gt_boxes = load_gt_boxes(label_path, width, height)

    result = m.predict(source=image, imgsz=IMGSZ, conf=0.001, iou=NMS_IOU, device=DEVICE, verbose=False )[0]

    if result.boxes is not None and len(result.boxes) > 0:
        all_pred_boxes = result.boxes.xyxy.cpu().numpy()
        all_confidences = result.boxes.conf.cpu().numpy()

        keep = all_confidences >= CONF_THRESHOLD

        pred_boxes = all_pred_boxes[keep]
        confidences = all_confidences[keep]
    else:
        pred_boxes = np.empty((0, 4), dtype=float)
        confidences = np.empty((0,), dtype=float)

    matches, fn_indices, fp_indices = match_predictions( gt_boxes,pred_boxes, match_iou=MATCH_IOU )

    tp = len(matches)
    fp = len(fp_indices)
    fn = len(fn_indices)

    records.append({"image": image_path.name,
                    "GT": len(gt_boxes),
                    "Predictions": len(pred_boxes),
                    "TP": tp,
                    "FP": fp,
                    "FN": fn})

    visual_data[image_path.name] = {"image_path": image_path,
                                    "gt_boxes": gt_boxes,
                                    "pred_boxes": pred_boxes,
                                    "confidences": confidences,
                                    "matches": matches,
                                    "fn_indices": fn_indices,
                                    "fp_indices": fp_indices,
                                    "tp": tp,
                                    "fp": fp,
                                    "fn": fn }

df = pd.DataFrame(records)

total_tp = int(df["TP"].sum())
total_fp = int(df["FP"].sum())
total_fn = int(df["FN"].sum())

precision = total_tp / (total_tp + total_fp) if total_tp + total_fp else 0
recall = total_tp / (total_tp + total_fn) if total_tp + total_fn else 0

print("\n========== FIXED-THRESHOLD ERROR ANALYSIS ==========")
print(f"Confidence threshold : {CONF_THRESHOLD}")
print(f"Matching IoU         : {MATCH_IOU}")
print(f"Images evaluated     : {len(df)}")
print(f"GT instances         : {int(df['GT'].sum())}")
print(f"TP                   : {total_tp}")
print(f"FP                   : {total_fp}")
print(f"FN                   : {total_fn}")
print(f"Precision            : {precision:.4f}")
print(f"Recall               : {recall:.4f}")

## Generate and save the test-set confusion matrix and precision-recall curve for YOLO26s.
# Run inference once, calculate TP/FP/FN across confidence thresholds, and visualize the results.

In [ ]:
# ======================================================================
# Notebook cell: confusion matrix + PR curve for the TEST set (YOLO26s)
# Needs what your notebook already defines:
#   m, image_paths, TEST_LABELS_DIR, IMGSZ, NMS_IOU, DEVICE,
#   CONF_THRESHOLD (0.1), MATCH_IOU (0.3), load_gt_boxes, match_predictions
# Writes: assets/confusion_matrix.png and assets/pr_curve.png
# ======================================================================
ASSETS_DIR = Path("assets")
ASSETS_DIR.mkdir(parents=True, exist_ok=True)

# ---- 1) Run inference ONCE at a very low confidence and keep every prediction ----
gt_by_image, preds_by_image = {}, {}

for image_path in image_paths:
    image = cv.imread(str(image_path))
    if image is None:
        print(f"Warning: could not read {image_path.name}")
        continue

    h, w = image.shape[:2]
    gt_by_image[image_path.name] = load_gt_boxes(TEST_LABELS_DIR / f"{image_path.stem}.txt", w, h)

    result = m.predict(source=image, imgsz=IMGSZ, conf=0.001, iou=NMS_IOU, device=DEVICE, verbose=False)[0]
    if result.boxes is not None and len(result.boxes) > 0:
        preds_by_image[image_path.name] = (result.boxes.xyxy.cpu().numpy(),
                                           result.boxes.conf.cpu().numpy())
    else:
        preds_by_image[image_path.name] = (np.empty((0, 4)), np.empty((0,)))


# ---- 2) TP / FP / FN at any confidence threshold, with the same matcher as before ----
def count_at(conf_thr, match_iou=MATCH_IOU):
    tp = fp = fn = 0
    for name, gt in gt_by_image.items():
        boxes, confs = preds_by_image[name]
        keep = confs >= conf_thr
        matches, fn_idx, fp_idx = match_predictions(gt, boxes[keep], match_iou=match_iou)
        tp += len(matches)
        fp += len(fp_idx)
        fn += len(fn_idx)
    return tp, fp, fn


tp, fp, fn = count_at(CONF_THRESHOLD)
precision = tp / (tp + fp) if tp + fp else 0.0
recall = tp / (tp + fn) if tp + fn else 0.0
print(f"conf={CONF_THRESHOLD}, IoU={MATCH_IOU}: TP={tp} FP={fp} FN={fn}  P={precision:.4f} R={recall:.4f}")
# -> must match the table in the README (TP 4734, FP 540, FN 129)


# ---- 3) Confusion matrix (same layout as Ultralytics: rows = predicted, columns = true) ----
def plot_confusion_matrix(tp, fp, fn, conf_thr, match_iou, out_path):
    #                  true helmet   true background
    matrix = np.array([[tp,          fp],           # predicted helmet
                       [fn,          np.nan]],      # predicted background (TN is undefined for detection)
                      dtype=float)
    labels = ["helmet", "background"]

    fig, ax = plt.subplots(figsize=(5.5, 4.8))
    shown = np.ma.masked_invalid(np.log10(matrix + 1))      # log colour scale: FN/FP stay visible next to TP
    ax.imshow(shown, cmap="Blues", vmin=0, vmax=np.log10(tp + 1) * 1.15)

    for i in range(2):
        for j in range(2):
            if np.isnan(matrix[i, j]):
                ax.text(j, i, "n/a", ha="center", va="center", color="gray", fontsize=12)
            else:
                dark = matrix[i, j] > tp * 0.1
                ax.text(j, i, f"{int(matrix[i, j]):,}", ha="center", va="center",
                        fontsize=15, fontweight="bold", color="white" if dark else "black")

    ax.set_xticks([0, 1], labels)
    ax.set_yticks([0, 1], labels)
    ax.set_xlabel("True")
    ax.set_ylabel("Predicted")
    ax.set_title(f"Test confusion matrix (conf ≥ {conf_thr}, IoU ≥ {match_iou})\n"
                 f"precision {tp / (tp + fp):.3f} · recall {tp / (tp + fn):.3f}", fontsize=11)
    fig.tight_layout()
    fig.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print("Saved:", Path(out_path).resolve())


plot_confusion_matrix(tp, fp, fn, CONF_THRESHOLD, MATCH_IOU, ASSETS_DIR / "confusion_matrix.png")


# ---- 4) Precision-Recall curve: sweep the confidence threshold (takes about a minute) ----
thresholds = np.unique(np.round(np.concatenate([[0.001, 0.005],
                                                np.arange(0.01, 0.1, 0.01),
                                                np.arange(0.1, 0.951, 0.025)]), 3))

curve = []
for t in thresholds:
    tp_t, fp_t, fn_t = count_at(float(t))
    if tp_t + fp_t == 0:
        continue
    curve.append((float(t), tp_t / (tp_t + fp_t), tp_t / (tp_t + fn_t)))
curve = np.array(curve)  # columns: conf, precision, recall


def plot_pr_curve(curve, op_conf, match_iou, out_path):
    conf, prec, rec = curve[:, 0], curve[:, 1], curve[:, 2]
    fig, ax = plt.subplots(figsize=(6, 4.8))
    ax.plot(rec, prec, color="tab:blue", lw=2, marker="o", ms=3)

    i = int(np.argmin(np.abs(conf - op_conf)))
    ax.scatter(rec[i], prec[i], s=80, color="tab:red", zorder=3)
    ax.annotate(f"conf = {conf[i]:.2f}\nP = {prec[i]:.3f}, R = {rec[i]:.3f}",
                (rec[i], prec[i]), textcoords="offset points", xytext=(-110, -45),
                fontsize=9, arrowprops=dict(arrowstyle="->", color="gray"))

    ax.set_xlabel("Recall")
    ax.set_ylabel("Precision")
    ax.set_xlim(max(0.0, rec.min() - 0.02), 1.0)
    ax.set_ylim(max(0.0, prec.min() - 0.02), 1.005)
    ax.grid(alpha=0.3)
    ax.set_title(f"Precision–Recall curve, test set (YOLO26s, IoU ≥ {match_iou})", fontsize=11)
    fig.tight_layout()
    fig.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print("Saved:", Path(out_path).resolve())


plot_pr_curve(curve, CONF_THRESHOLD, MATCH_IOU, ASSETS_DIR / "pr_curve.png")

## Draw GT boxes (green, or thick yellow if missed = FN) and predictions (blue = TP, red = FP) with labels on one image; show_examples plots a random sample of images that contain TP, FP, or FN cases

In [ ]:
def draw_error_image(info):
    image = cv.imread(str(info["image_path"]))
    image = cv.cvtColor(image, cv.COLOR_BGR2RGB)

    # Draw all GT boxes in green.
    for gt_idx, box in enumerate(info["gt_boxes"]):
        x1, y1, x2, y2 = map(int, box)

        color = (0, 255, 0)
        thickness = 1

        if gt_idx in info["fn_indices"]:
            color = (255, 255, 0)
            thickness = 3

        cv.rectangle(image, (x1, y1), (x2, y2), color, thickness )
        cv.putText(image, f"GT {gt_idx + 1}", (x1, max(15, y1 - 5)), cv.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)

    matched_pred_indices = {pred_idx for _, pred_idx, _ in info["matches"] }

    # Draw TP predictions in blue and FP predictions in red.
    for pred_idx, box in enumerate(info["pred_boxes"]):
        x1, y1, x2, y2 = map(int, box)
        conf = info["confidences"][pred_idx]

        if pred_idx in matched_pred_indices:
            color = (0, 150, 255)  # Blue in RGB
            label = f"TP {conf:.2f}"
        else:
            color = (255, 0, 0)
            label = f"FP {conf:.2f}"

        cv.rectangle( image, (x1, y1), (x2, y2), color, 2)
        cv.putText( image, label, (x1, min(image.shape[0] - 5, y2 + 15)), cv.FONT_HERSHEY_SIMPLEX, 0.5, color, 1 )

    return image



def show_examples(category=None, names=None, n=3, seed=42):
    """Show images containing TP/FP/FN cases, or an explicit list of image names."""

    if names is None:
        category = category.upper()
        if category not in {"TP", "FP", "FN"}:
            raise ValueError("category must be TP, FP, or FN")
        candidates = [name for name, info in visual_data.items() if info[category.lower()] > 0]
        if not candidates:
            print(f"No images containing {category} were found.")
            return
        rng = random.Random(seed)
        selected = rng.sample(candidates, min(n, len(candidates)))
    else:
        selected = list(names)[:n]

    fig, axes = plt.subplots(1, len(selected), figsize=(7 * len(selected), 6), squeeze=False)

    for ax, name in zip(axes[0], selected):
        info = visual_data[name]
        ax.imshow(draw_error_image(info))
        ax.set_title(f"{name}\nTP={info['tp']} | FP={info['fp']} | FN={info['fn']}", fontsize=9)
        ax.axis("off")

    plt.tight_layout()

## Show random TP examples and the worst FP / FN images from the test set

In [ ]:
print("========== TP EXAMPLES ==========")
show_examples("TP", n=3)

print("========== WORST FP EXAMPLES ==========")
worst_fp = df.sort_values("FP", ascending=False).head(3)["image"].tolist()
show_examples(names=worst_fp)

print("========== WORST FN EXAMPLES ==========")
worst_fn = df.sort_values("FN", ascending=False).head(3)["image"].tolist()
show_examples(names=worst_fn)

In [ ]:
show_examples('TP' , n=6)
plt.savefig("sample_predictions.png", dpi=150, bbox_inches="tight")

In [ ]:
# Combine FN and FP examples into one figure.
selected = worst_fn + worst_fp
fig, axes = plt.subplots(2, 3, figsize=(18, 12))

for row, names in enumerate([worst_fn, worst_fp]):
    for col, name in enumerate(names):
        info = visual_data[name]
        ax = axes[row, col]
        ax.imshow(draw_error_image(info))
        ax.set_title( f"{name}\nTP={info['tp']} | FP={info['fp']} | FN={info['fn']}", fontsize=9 )
        ax.axis("off")

# Add row labels to distinguish FN and FP examples.
fig.text(0.02, 0.73, "FN Examples", rotation=90, va="center", fontsize=13)
fig.text(0.02, 0.28, "FP Examples", rotation=90 , va="center", fontsize=13)
plt.tight_layout(rect=[0.04, 0, 1, 1])
plt.savefig('assets/error_examples.png' , dpi=150 , bbox_inches="tight")

## Load the test split into supervision and get class names

In [ ]:
test_ds=sv.DetectionDataset.from_yolo(images_directory_path='/mnt/d/dataset/helmet_dataset/test/images',
                                 annotations_directory_path='/mnt/d/dataset/helmet_dataset/test/labels',
                                 data_yaml_path='/mnt/d/dataset/helmet_dataset/data.yaml')
class_names=test_ds.classes

## Run inference on the test split (conf=0.1), build per-image confusion matrices (IoU=0.5), and collect images with FN or FP, ranked by error count

In [ ]:
test_predictions, test_labels, test_img_pathes = [], [], []
    
for img_path, img, label in test_ds:
    predict = m.predict(img, imgsz=640, conf=0.1, iou=0.5, device=0, save=False, verbose=False)[0]
    
    detection = sv.Detections.from_ultralytics(predict).with_nms()
    
    test_predictions.append(detection)
    test_labels.append(label)
    test_img_pathes.append(img_path)
    
    
# Image names
test_image_names = [Path(p).name for p in test_img_pathes]
    
    
    
    
error_images_test = {}
    
for name, pred_det, gt_det in zip(test_image_names, test_predictions, test_labels):
    cm = sv.ConfusionMatrix.from_detections( predictions=[pred_det], targets=[gt_det], classes=class_names, conf_threshold=0.1, iou_threshold=0.3 )
    
    tp = int(cm.matrix[0, 0])
    fn = int(cm.matrix[0, 1])
    fp = int(cm.matrix[1, 0])
    
    if fp > 0 or fn > 0:
        error_images_test[name] = { "TP": tp, "FP": fp, "FN": fn}
    
    
    
# -------------------------
# FN images
# -------------------------
    
fn_images_test = { name: result["FN"] for name, result in error_images_test.items() if result["FN"] > 0}
    
sorted_fn_test = sorted( fn_images_test.items(), key=lambda x: x[1], reverse=True)
    
    
# -------------------------
# FP images
# -------------------------
    
fp_images_test = { name: result["FP"] for name, result in error_images_test.items() if result["FP"] > 0}
    
sorted_fp_test = sorted( fp_images_test.items(), key=lambda x: x[1], reverse=True)
    
    
print(f"تعداد کل عکس‌هایی که FN دارن: {len(fn_images_test)}")
print(f"تعداد کل عکس‌هایی که FP دارن: {len(fp_images_test)}")

## Draw TP predictions (green), FP predictions (red), and missed GT boxes (FN, yellow) on a copy of the image

In [ ]:
def matched_indices(gt_xyxy, pred_xyxy, iou_threshold=0.3):
    """برمی‌گردونه: اندیس‌های GT و prediction که match شدن"""
    matched_gt, matched_pred = set(), set()
    if len(gt_xyxy) and len(pred_xyxy):
        iou_matrix = box_iou(gt_xyxy, pred_xyxy).clone()
        
        for _ in range(min(iou_matrix.shape)):
            best_iou, idx = iou_matrix.max(), iou_matrix.argmax()
            
            if best_iou < iou_threshold:
                break
            
            gt_idx, pred_idx = divmod(idx.item(), iou_matrix.shape[1])
            matched_gt.add(gt_idx)
            matched_pred.add(pred_idx)
            iou_matrix[gt_idx, :] = -1
            iou_matrix[:, pred_idx] = -1
   
    return matched_gt, matched_pred

def draw_errors(image, pred_det, gt_det, iou_threshold=0.3):
    matched_gt, matched_pred = matched_indices(torch.tensor(gt_det.xyxy, dtype=torch.float32),
                                               torch.tensor(pred_det.xyxy, dtype=torch.float32), iou_threshold)
    
    tp_mask = np.isin(np.arange(len(pred_det)), list(matched_pred))
    fn_mask = ~np.isin(np.arange(len(gt_det)), list(matched_gt))

    image = sv.BoxAnnotator(color=sv.Color.GREEN, thickness=1).annotate(image.copy(), pred_det[tp_mask])
    image = sv.BoxAnnotator(color=sv.Color.RED, thickness=1).annotate(image, pred_det[~tp_mask])
    image = sv.BoxAnnotator(color=sv.Color.YELLOW, thickness=1).annotate(image, gt_det[fn_mask])
    return image

## Show test images with FP in 2x2 grids with TP/FP/FN boxes and counts

In [ ]:
images_to_show = list(fp_images_test) 

for start in range(0, len(images_to_show), 4):
    batch = images_to_show[start:start + 4]
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))

    for ax, name in zip(axes.flat, batch):
        idx = test_image_names.index(name)

        image_bgr = cv.imread(Path(TEST_IMAGES_DIR/ name))
        annotated_bgr = draw_errors(image_bgr, test_predictions[idx], test_labels[idx])
        annotated_rgb = cv.cvtColor(annotated_bgr, cv.COLOR_BGR2RGB)

        result = error_images_test[name]
        ax.imshow(annotated_rgb)
        ax.set_title(f"{name}\nTP: {result['TP']} | FP: {result['FP']} | FN: {result['FN']}", fontsize=9)
        ax.axis('off')

    for ax in axes.flat[len(batch):]:
        ax.axis('off')

    plt.tight_layout()
    plt.show()

## Save per-image error counts and overall model evaluation metrics as CSV files.

In [ ]:
OUTPUT_DIR = Path("final_helmet_evaluation")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

df.to_csv( OUTPUT_DIR / "test_error_counts.csv", index=False)

summary = pd.DataFrame([{"model": m,
                         "confidence_threshold": CONF_THRESHOLD,
                         "nms_iou": NMS_IOU,
                         "matching_iou": MATCH_IOU,
                         "images": len(df),
                         "GT": int(df["GT"].sum()),
                         "TP": total_tp,
                         "FP": total_fp,
                         "FN": total_fn,
                         "precision": precision,
                         "recall": recall,
                         "mAP50": test_results.box.map50,
                         "mAP50_95": test_results.box.map}])

summary.to_csv(OUTPUT_DIR / "final_metrics.csv", index=False)

print(f"Results saved to: {OUTPUT_DIR.resolve()}")